# SmartEval — Handwriting Extraction Test

**Purpose:** find out how well handwriting is actually recognised from real answer-book photos, before building anything else.

**Scope:** extraction only. No answer key, no scoring, no evaluation logic.

---

### What this notebook does

1. Loads **TrOCR** (Microsoft's handwriting model) and **EasyOCR** (line detector + fallback)
2. Takes photos of answer-book pages
3. Detects text lines, crops them, runs TrOCR on each crop
4. Shows the transcript, per-line confidence, and timing
5. Saves everything to CSV + JSON

### Before you start

- `Runtime → Change runtime type → T4 GPU` (CPU works, but ~10x slower)
- Have **5 real answer-book photos** ready — shot on a phone, the way a lecturer would
- First run takes ~5 minutes (TrOCR downloads 1.3 GB)

### The one thing to understand first

TrOCR reads **one text line at a time**. It has no idea what a page is.

Cell 9 runs TrOCR on a whole answer block *on purpose*, to show you what that failure looks like. That's why the pipeline detects lines first. Don't skip it — it's the justification for everything that follows.

## 1. Environment check

In [ ]:
import sys, platform, torch

print("Python  :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("Platform:", platform.platform())

if torch.cuda.is_available():
    print("Device  : GPU —", torch.cuda.get_device_name(0))
else:
    print("Device  : CPU only")
    print("\n⚠️  No GPU detected. Everything still works, just much slower.")
    print("   Runtime → Change runtime type → T4 GPU to speed it up.")

## 2. Install dependencies

⚠️ **Do not upgrade `pandas` or `pillow`.** Colab ships specific versions of both, and Colab's own packages (`gradio`, `dask`, `cudf`) pin them. Upgrading produces dependency-conflict warnings and can break other Colab features.

`easyocr` pulls in what it needs on its own.

In [ ]:
%pip install -q "transformers>=4.40" easyocr

In [ ]:
# Sanity check — these should match what Colab shipped with.
import pandas, PIL, torch, transformers

print("pandas       :", pandas.__version__)
print("pillow       :", PIL.__version__)
print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)

## 3. Configuration

`MODEL_NAME` — switch to `microsoft/trocr-large-handwritten` if base quality is not enough (4× slower, ~2 GB).

`CROP_PAD` — pixels of padding around each detected line. Too tight and TrOCR loses ascenders/descenders.

In [ ]:
from pathlib import Path
import time, json
import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display

MODEL_NAME = "microsoft/trocr-base-handwritten"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN    = 64      # generous; the default truncates long lines
CROP_PAD   = 4       # px padding around each detected line

UPLOAD_DIR = Path("/content/uploads")
UPLOAD_DIR.mkdir(exist_ok=True)

print(f"{MODEL_NAME}  →  {DEVICE}")

## 4. Load models

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

print("✅ TrOCR loaded")

EasyOCR is used for **two** things: finding line boxes, and acting as a fallback recogniser when TrOCR produces garbage.

In [ ]:
import easyocr

reader = easyocr.Reader(["en"], gpu=(DEVICE == "cuda"))
print("✅ EasyOCR loaded")

## 5. Upload your photos

Run the cell and pick 5 images. They land in `/content/uploads`.

**Alternative — Google Drive:**

```python
from google.colab import drive
drive.mount("/content/drive")
UPLOAD_DIR = Path("/content/drive/MyDrive/answer_scripts")
```

In [ ]:
from google.colab import files

uploaded = files.upload()

for name, data in uploaded.items():
    (UPLOAD_DIR / name).write_bytes(data)

imgs = sorted(p for p in UPLOAD_DIR.iterdir()
              if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print(f"\n{len(imgs)} image(s) ready:")
for p in imgs:
    print(" •", p.name)

## 6. Core functions

Three pieces:

- `trocr_line(img)` — one image → text + confidence + seconds
- `detect_lines(path)` — EasyOCR finds and sorts line boxes top-to-bottom
- `is_degenerate(text)` — catches TrOCR's repetition/hallucination failure mode

Confidence here is `exp(mean(token_logprobs))` — a **heuristic**, not a calibrated probability. Never describe it as one.

In [ ]:
def trocr_line(img: Image.Image):
    # Run TrOCR on a single-line crop.
    px = processor(images=img.convert("RGB"), return_tensors="pt").pixel_values.to(DEVICE)

    t0 = time.time()
    out = model.generate(
        px, max_length=MAX_LEN,
        output_scores=True, return_dict_in_generate=True,
    )
    dt = time.time() - t0

    text = processor.batch_decode(out.sequences, skip_special_tokens=True)[0].strip()

    probs = [torch.softmax(s, dim=-1) for s in out.scores]
    toks  = out.sequences[0, 1:]
    if len(toks):
        conf = float(torch.stack([probs[i][0, t] for i, t in enumerate(toks)]).mean())
    else:
        conf = 0.0

    return text, conf, dt


def is_degenerate(text: str) -> bool:
    # TrOCR repeats or hallucinates rather than erroring. Catch that.
    words = text.lower().split()
    if len(text.strip()) < 2:
        return True
    if len(words) >= 6:
        trigrams = [" ".join(words[i:i + 3]) for i in range(len(words) - 2)]
        if len(set(trigrams)) < len(trigrams) * 0.6:
            return True
    return False


def detect_lines(path):
    # EasyOCR boxes, sorted top-to-bottom.
    res = reader.readtext(str(path), detail=1, paragraph=False)
    boxes = []
    for pts, text, conf in res:
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        boxes.append({
            "x1": int(min(xs)), "y1": int(min(ys)),
            "x2": int(max(xs)), "y2": int(max(ys)),
            "easyocr_text": text,
            "easyocr_conf": float(conf),
        })
    # bucket y to tolerate slight skew, then sort left-to-right within a band
    boxes.sort(key=lambda b: (b["y1"] // 12, b["x1"]))
    return boxes


print("✅ helpers defined")

## 7. The pipeline

```
photo → EasyOCR detects lines → crop each (with padding) → TrOCR → join
                                      ↓
                            degenerate output?
                                      ↓ yes
                            use EasyOCR's own text
```

In [ ]:
def extract(path):
    path = Path(path)
    img = Image.open(path).convert("RGB")
    t_start = time.time()

    # (A) WRONG WAY — whole image through TrOCR. Kept for comparison only.
    block_text, block_conf, _ = trocr_line(img)

    # (B) CORRECT WAY — detect lines, crop, recognise each
    lines = []
    for b in detect_lines(path):
        crop = img.crop((
            max(0, b["x1"] - CROP_PAD),
            max(0, b["y1"] - CROP_PAD),
            min(img.width,  b["x2"] + CROP_PAD),
            min(img.height, b["y2"] + CROP_PAD),
        ))

        txt, conf, dt = trocr_line(crop)
        used_fallback = False

        if is_degenerate(txt):
            txt, conf, used_fallback = b["easyocr_text"], b["easyocr_conf"], True

        lines.append({
            "box": [b["x1"], b["y1"], b["x2"], b["y2"]],
            "trocr_text": txt,
            "trocr_conf": round(conf, 3),
            "easyocr_text": b["easyocr_text"],
            "seconds": round(dt, 2),
            "used_fallback": used_fallback,
        })

    return {
        "file": path.name,
        "line_count": len(lines),
        "pipeline_text": "\n".join(l["trocr_text"] for l in lines),
        "pipeline_conf": round(float(np.mean([l["trocr_conf"] for l in lines])), 3) if lines else 0.0,
        "fallback_lines": sum(l["used_fallback"] for l in lines),
        "whole_image_text": block_text,
        "whole_image_conf": round(block_conf, 3),
        "seconds": round(time.time() - t_start, 2),
        "lines": lines,
    }


def show(result):
    print("=" * 78)
    print(f"{result['file']}   |   {result['line_count']} lines   |   "
          f"conf {result['pipeline_conf']}   |   {result['seconds']}s")
    print("=" * 78)
    print("\n--- PROPER PIPELINE (detect → crop → TrOCR) ---\n")
    print(result["pipeline_text"] or "(nothing extracted)")
    print("\n--- WHOLE IMAGE THROUGH TrOCR (expected to be bad) ---\n")
    print(result["whole_image_text"] or "(nothing)")


print("✅ pipeline defined")

## 8. Try it on the first image

Look at both blocks. The first should read your handwriting; the second should demonstrate why line detection is mandatory.

In [ ]:
imgs = sorted(p for p in UPLOAD_DIR.iterdir()
              if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})

first = extract(imgs[0])
show(first)

print("\n--- per line ---")
for i, l in enumerate(first["lines"], 1):
    fb = " [fallback]" if l["used_fallback"] else ""
    print(f"{i:2d}. ({l['trocr_conf']:.2f}) {l['trocr_text']}{fb}")

## 9. TrOCR at its best — one hand-cropped line

This is what the model can do when given a clean single line. Adjust the crop box to match a line in your photo.

Compare this to the pipeline output above. The gap is *your* detection quality, not the model's.

In [ ]:
img = Image.open(imgs[0]).convert("RGB")
w, h = img.size

# 👇 change these fractions to frame one line in your image
crop = img.crop((int(w * 0.05), int(h * 0.10), int(w * 0.95), int(h * 0.19)))

display(crop.resize((crop.width * 2, crop.height * 2)))

text, conf, dt = trocr_line(crop)
print(f"\nTrOCR : {text}")
print(f"conf  : {conf:.2f}   ({dt:.1f}s)")

## 10. Run over every uploaded image

In [ ]:
results = [extract(p) for p in imgs]

print(f"\n✅ processed {len(results)} image(s)")

## 11. Summary

In [ ]:
df = pd.DataFrame([{
    "file": r["file"],
    "lines": r["line_count"],
    "chars": len(r["pipeline_text"]),
    "confidence": r["pipeline_conf"],
    "fallback_lines": r["fallback_lines"],
    "seconds": r["seconds"],
} for r in results])

display(df)

print(f"Mean confidence        : {df.confidence.mean():.3f}")
print(f"Mean seconds / image   : {df.seconds.mean():.1f}")
print(f"Mean seconds / line    : {df.seconds.sum() / max(df.lines.sum(), 1):.1f}")
print(f"Fallback rate          : {df.fallback_lines.sum() / max(df.lines.sum(), 1):.1%}")

## 12. Full transcripts

In [ ]:
for r in results:
    show(r)
    print("\n")

## 13. Save and download

Keep these files — the transcript dump plus the confidence numbers are the first real table in your report.

In [ ]:
pd.DataFrame([{k: v for k, v in r.items() if k != "lines"} for r in results]) \
  .to_csv("/content/ocr_results.csv", index=False)

with open("/content/ocr_results.json", "w") as f:
    json.dump(results, f, indent=2)

from google.colab import files as colab_files
colab_files.download("/content/ocr_results.csv")
colab_files.download("/content/ocr_results.json")

print("✅ saved")

## 14. Reading the results

### Confidence

Rough bands for `pipeline_conf`:

| Confidence | Meaning |
|---|---|
| **> 0.80** | Text is very likely close to correct |
| **0.60 – 0.80** | Mostly right, expect a few words wrong |
| **< 0.60** | Treat with suspicion — this is where the review flag fires |

### Eyeball accuracy

Without ground truth you can still judge quickly:

1. Read the transcript out loud against the photo.
2. Count how many words are wrong per line.
3. Roughly: **1 wrong word in 10 ≈ 10% CER.**

If you want a real number, add `jiwer` and compare against hand-typed reference text:

```python
!pip -q install jiwer
import jiwer
print(jiwer.cer("dijkstra finds shortest path", "dijkstra finds shortest paths"))
```

### What you're looking for

| Observation | Meaning |
|---|---|
| 4–6 lines detected on a 6-line answer | Detection is working |
| One giant line covering the whole page | Detector failed — usually low resolution or heavy skew |
| TrOCR repeats the same phrase | Degenerate output — fallback should have caught it |
| Lots of `[fallback]` lines | EasyOCR is carrying the pipeline; investigate TrOCR crops |
| `whole_image_text` looks like line 1 repeated | Expected. This is the failure we designed around. |

---

## 15. Troubleshooting

| Problem | Fix |
|---|---|
| CUDA out of memory | Restart runtime; use `-base` not `-large`; process one image at a time |
| Very slow | Runtime type is CPU — switch to T4 GPU |
| Empty transcripts | Image too small. Shoot at higher resolution; lines under ~20 px tall fail |
| Everything merged into one line | Increase photo resolution, avoid extreme angles |
| Model download stalls | Re-run the cell; Hugging Face occasionally throttles |
| `easyocr` install fails | Restart runtime, then re-run the install cell first |
| pip warns about `pandas`/`pillow` conflicts | You upgraded them. `Runtime → Disconnect and delete runtime` gives a fresh VM with stock versions. Restart alone keeps the upgraded packages |

---

## 16. What to do with the result

Come back with three numbers:

1. **Mean confidence** across your 5 photos
2. **Mean seconds per image** (this sizes your async job design)
3. **Your eyeballed error rate** per line

If confidence is above ~0.75 and the transcripts are readable, the project is safe — build the rest.

If it's below 0.60, don't build a UI yet. Try `trocr-large-handwritten`, then better line cropping, before writing a single screen.